# 01 — Data Understanding

## Project objective
Understand the UCI Online Retail II transaction dataset before cleaning or modelling. The eventual business task is to estimate whether a customer makes at least one valid purchase in the **90 days after a monthly snapshot**, using only the preceding **180 days** of behaviour.

This is a repeat-purchase propensity problem—not a directly observed contractual churn problem.

## Dataset and handling
- Source: [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail%2Bii)
- Expected local file: `data/raw/online_retail_II.xlsx`
- The workbook is not committed to Git. Download it from UCI before running the data cells.
- The source is licensed CC BY 4.0; retain attribution when reusing it.

## Learning goals
1. Inspect workbook sheets, dimensions, columns and data types.
2. Profile missing values, duplicates, date coverage and numeric ranges.
3. Identify cancellation invoices and non-purchase rows.
4. Record data-quality decisions before building customer features.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Find the project root even when Jupyter starts in the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data.py").exists():
    raise RuntimeError("Open Jupyter from the project folder or one of its subfolders.")
sys.path.insert(0, str(PROJECT_ROOT))

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
print("Project root:", PROJECT_ROOT)
print("Workbook exists:", RAW_FILE.exists())

## 1. Inspect the workbook

The official workbook is expected to contain two annual worksheets. We inspect sheet names and dimensions without loading the full workbook into a single DataFrame first.

In [ ]:
if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Download the UCI workbook and save it here: {RAW_FILE}"
    )

excel = pd.ExcelFile(RAW_FILE, engine="openpyxl")
print("Sheets:", excel.sheet_names)
for sheet in excel.sheet_names:
    preview = pd.read_excel(excel, sheet_name=sheet, nrows=5)
    print(f"\\n{sheet}: preview shape={preview.shape}")
    display(preview)

## 2. Load and inspect source columns

The reusable ingestion module standardizes the source column names. At this stage we inspect the combined raw rows; cleaning decisions are documented in the next notebook.

In [ ]:
from src.data import load_workbook

raw = load_workbook(RAW_FILE)
print("Combined shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())
raw.info()

## 3. Data-quality profile

Check missingness, duplicate rows, date range, quantity and unit-price distributions. Do not silently discard cancellation rows: they are useful for returns analysis even though they are excluded from purchase modelling.

In [ ]:
quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique(dropna=True),
})
display(quality)
print("Exact duplicate rows:", int(raw.duplicated().sum()))

raw_dates = pd.to_datetime(raw["InvoiceDate"], errors="coerce")
print("Invoice date range:", raw_dates.min(), "to", raw_dates.max())
print("Quantity summary:")
display(pd.to_numeric(raw["Quantity"], errors="coerce").describe())
print("Price summary:")
display(pd.to_numeric(raw["Price"], errors="coerce").describe())

## 4. Cancellation and invalid-value checks

Invoice numbers beginning with `C` identify cancellation lines in this dataset. Negative or zero quantities/prices should be investigated, not interpreted as ordinary purchases.

In [ ]:
invoice_text = raw["Invoice"].astype("string").str.strip()
quantity = pd.to_numeric(raw["Quantity"], errors="coerce")
price = pd.to_numeric(raw["Price"], errors="coerce")
checks = pd.Series({
    "cancellation_lines": invoice_text.str.upper().str.startswith("C", na=False).sum(),
    "missing_customer_id": raw["Customer ID"].isna().sum(),
    "non_positive_quantity": quantity.le(0).sum(),
    "non_positive_price": price.le(0).sum(),
})
display(checks.to_frame("row_count"))

## 5. Record your findings

Before moving on, answer these questions in your project notes:
- Which fields have the most missing values, and why might Customer ID be absent?
- What proportion of rows are cancellation lines?
- Are there negative or zero quantities/prices, and what do they represent?
- Do the two worksheets have consistent schemas?
- What checks should be repeated after cleaning?

**Important:** This notebook profiles the actual workbook only when you run it locally. No dataset counts or findings are pre-filled.